# ML-04 - Search Intelligence Data Contract

This notebook keeps the contract short, visible, and testable. Each claim is backed by a query output.


## 1. Unit of analysis + time window

- Daily rows are aggregated to one row per client × content item for the March 2026 month.
- Tables: the March partition of `fact_content_daily_performance`, joined to `dim_content` and `dim_clients`.
- Time window: month=2026-03 is the current review window. This is a contract check for the scoreable slice, not a future-outcome forecast.
- Label / proxy: the lane is CTR / Engagement Opportunity Scoring, but GA4 engagement data is only partly available, so many rows have zero GA4 sessions or zero engaged sessions. This project will mainly rank pages by CTR gap and later add a decline outcome.
- Deliberate exclusion: raw URL, keyword, query, and product-decision fields are excluded because they are not part of the scoreable evidence set.


## 2. Fields: feature / label / context / excluded

- Feature: `impressions_month` - knowable at the decision moment because it is the month-total impression count observed before the decision point.
- Feature: `avg_position_month` - knowable at the decision moment because it is the simple mean of daily positions greater than 0, without weighting by impressions.
- Feature: `days_with_data` - knowable at the decision moment because it counts the days with observed impressions inside the same month.
- Feature: `content_age_days` - knowable at the decision moment because it is computed from the content creation date and fixed before the decision.
- Feature: `position_volatility` - knowable at the decision moment because it is the standard deviation of daily `gsc_avg_position` values above zero in the same month.
- Label / proxy: `low_ctr_flag` is built from `ctr_month`; `ctr_month` stays in the label definition, never in the feature set.
- Context: `client_hash_id`, `content_hash_id`.
- Excluded: raw query, URL, keyword, and product-decision columns are excluded because they are not part of the scoreable evidence set.
- Proxy note: the label is a same-month rule, not an observed future outcome. Later weeks replace it with future-window outcomes.


In [6]:
import os
import getpass
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'work').is_dir() and (p / 'skills').is_dir()
)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    env_path = root / '.env'
    if env_path.exists():
        for line in env_path.read_text(encoding='utf-8').splitlines():
            if line.strip().startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            if key.strip() == 'HF_TOKEN':
                HF_TOKEN = value.strip().strip('\"').strip("'")
                break
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is missing. Please enter a valid Hugging Face READ token.')

os.environ['HF_TOKEN'] = HF_TOKEN

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
month = '2026-03'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={month}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

base = con.sql(f'''
    SELECT f.report_date, f.client_hash_id, f.content_hash_id,
           f.gsc_impressions, f.gsc_clicks, f.gsc_avg_position,
           f.ga4_data_available, f.ga4_sessions, f.ga4_engaged_sessions,
           c.content_created_date
    FROM {FACT} AS f
    LEFT JOIN {DIM_CONTENT} AS c
      ON f.content_hash_id = c.content_hash_id
     AND f.client_hash_id = c.client_hash_id
''').df()

base['ctr_month'] = np.where(base['gsc_impressions'].fillna(0) > 0, 100.0 * base['gsc_clicks'] / base['gsc_impressions'], np.nan)
base['page_has_engagement'] = base['ga4_engaged_sessions'].fillna(0) > 0
print('rows:', len(base))

per_page = base[base['gsc_impressions'].fillna(0) > 0].copy()
per_page['page_ctr'] = 100.0 * per_page['gsc_clicks'] / per_page['gsc_impressions']
weighted_ctr = 100.0 * per_page['gsc_clicks'].sum() / per_page['gsc_impressions'].sum()
mean_page_ctr = per_page['page_ctr'].mean()
print('mean_daily_ctr_pct:', round(mean_page_ctr, 3))
print('weighted_ctr_pct:', round(weighted_ctr, 3))
print('weighted_ctr_explained:', 'Weighted CTR uses total clicks over total impressions, so it differs from the simple mean of page CTR values.')


rows: 9841378
mean_daily_ctr_pct: 0.308
weighted_ctr_pct: 0.293
weighted_ctr_explained: Weighted CTR uses total clicks over total impressions, so it differs from the simple mean of page CTR values.


The data cell reports 9,841,378 daily rows, a mean daily CTR of 0.308%, and a weighted CTR of 0.293%. The weighted value uses total clicks divided by total impressions, while the mean gives each page-day equal weight.


In [7]:
grain_violations = con.sql(f'''
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n_rows
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    ORDER BY n_rows DESC
    LIMIT 5
''').df()
print('grain_violations:', len(grain_violations))
if len(grain_violations):
    print(grain_violations.to_string(index=False))
else:
    print('No grain violations in the March slice.')


grain_violations: 0
No grain violations in the March slice.


This proves the March fact table has one daily row per client-content-date combination in the current slice; any value above zero would mean a repeated row at the grain we rely on.


In [8]:
summary = con.sql(f'''
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT}
''').df()
print(summary.to_string(index=False))


 n_rows  n_content  n_clients   min_date   max_date
9841378     331437         55 2026-03-01 2026-03-31


This proves the selected slice is internally consistent in size and date coverage before any modeling step.


In [9]:
availability = con.sql(f'''
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_that_survive,
           100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*) AS pct_available
    FROM {FACT}
''').df()
print(availability.to_string(index=False))


 total_rows  rows_that_survive  pct_available
    9841378           413966.0       4.206382


This proves the share of March rows with usable GA4 coverage. It shows whether the engagement-side availability is strong enough for the ranking lane or whether the GA4 view remains partial.


In [10]:
g = base[base['gsc_impressions'].fillna(0) > 0].copy()
g['pos_valid'] = g['gsc_avg_position'].where(g['gsc_avg_position'] > 0)

page = (g.groupby(['client_hash_id', 'content_hash_id'])
          .agg(impressions_month=('gsc_impressions', 'sum'),
               clicks_month=('gsc_clicks', 'sum'),
               avg_position_month=('pos_valid', 'mean'),
               days_with_data=('report_date', 'nunique'),
               position_volatility=('pos_valid', lambda s: s.std(ddof=0)),
               created=('content_created_date', 'first'))
          .reset_index())
page['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(page['created'])).dt.days
page['ctr_month'] = 100.0 * page['clicks_month'] / page['impressions_month']
page = page.dropna(subset=['avg_position_month']).copy()

page['position_tier'] = pd.cut(page['avg_position_month'], bins=[0, 3, 10, 20, 50, np.inf],
                               labels=['top_3', 'page_1', 'striking', 'page_3_5', 'deep'])
elig = page['impressions_month'] >= 100
tot = page[elig].groupby('position_tier', observed=True)[['clicks_month', 'impressions_month']].sum()
expected = 100.0 * tot['clicks_month'] / tot['impressions_month']
page['tier_expected_ctr'] = page['position_tier'].map(expected).astype(float)
page['low_ctr_proxy'] = np.where(elig, (page['ctr_month'] < page['tier_expected_ctr']).astype(float), np.nan)

FEATURES = ['impressions_month', 'avg_position_month', 'days_with_data', 'content_age_days', 'position_volatility']
print(page[FEATURES].head().to_string(index=False))

def precision_at_k(score, label, k):
    s, y = np.asarray(score, float), np.asarray(label, float)
    m = ~(np.isnan(s) | np.isnan(y))
    order = np.argsort(-s[m], kind='mergesort')[:k]
    return float(y[m][order].mean())

rank_df = page[page['low_ctr_proxy'].notna()]
assert rank_df['low_ctr_proxy'].nunique() == 2
print('labelled pages:', len(rank_df), '| base rate:', round(rank_df['low_ctr_proxy'].mean(), 3))
print('honest P@20 (position only):', round(precision_at_k(rank_df['avg_position_month'], rank_df['low_ctr_proxy'], 20), 3))
print('LEAKY P@20 (+ ctr_month):   ', round(precision_at_k(-rank_df['ctr_month'], rank_df['low_ctr_proxy'], 20), 3))
print('honest P@20 after removal:  ', round(precision_at_k(rank_df['avg_position_month'], rank_df['low_ctr_proxy'], 20), 3))

 impressions_month  avg_position_month  days_with_data  content_age_days  position_volatility
                 1            9.000000               1               175             0.000000
               331           14.129210              31               175             4.886748
                33            9.225529               6               175             0.722158
               145            8.470926              30               175             1.326703
               461           14.859827              31               175             4.180947
labelled pages: 101441 | base rate: 0.691
honest P@20 (position only): 0.9
LEAKY P@20 (+ ctr_month):    1.0
honest P@20 after removal:   0.9


The deliberate leak is shown in the code above: first the honest baseline with a score that ignores zero-position rows, then the leaky score that adds `ctr_month`, and then the honest result again after the leaked score is removed. The leak is not a real signal; it is the same-month label being used as a ranking score.

The run had 101,441 labelled pages with a base rate of 0.691. Honest P@20 was 0.90, the leaky P@20 was 1.00, and honest P@20 after removing the leak was 0.90.


## 4. Data limits

**Limit 1 - unbalanced panel / GA4 zero-fill:** GA4 was available in 413,966 of 9,841,378 rows (4.2%). Many rows have zero GA4 sessions or zero engaged sessions. Some of that is real, and some is a data-availability issue, so the engagement view is not a complete source of truth for every page.

**Limit 2 - same-month features and proxy label:** the features and the proxy label come from the same month, so this is a data-contract exercise, not a forecast. Week 4 fixes this by using March-May features and a June outcome.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Each section is backed by a visible query or printed result
- [ ] The notebook runs top to bottom with no errors
- [ ] No client names, URLs, or raw queries are in the notebook-facing text
- [ ] The wording stays measured, directional, and decision-support
- [ ] This is a contract check, not a causal claim about what a content edit will do
